# 02 Stop-Loss Benchmark

For each candidate stop loss, calculates the after-tax value if stopped and the required recovery from that stop price to match selling today.

**Plain English:**
This notebook shows how much damage each stop level causes, and how much the price must bounce back from that lower level before holding would catch up to selling today.

**This answers the question:** "How deep can I let the price fall before the required comeback becomes too large?"

Example:
A 20% stop from 350 triggers at 280. To get back to the sell-today benchmark of 350, the price must rebound 25% from 280.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tax_risk_sim import (
    build_after_tax_sensitivity,
    build_bear_recovery_cases,
    build_bear_recovery_table,
    build_probability_weighted_scenarios,
    build_stop_benchmark,
    compare_stop_reentry_vs_hold,
    sell_today_baseline,
)

pd.options.display.float_format = "{:,.2f}".format

## Inputs

These are the position assumptions and the stop-loss levels to test.

**Plain English:**
Change the position numbers and the candidate stop percentages here.

**This answers the question:** "Which stop levels should I compare?"

Example:
`stop_loss_drops = [0.05, 0.10, 0.20]` tests 5%, 10%, and 20% stops.

### Input Explanations

`shares`, `current_price`, `cost_basis_per_share`, and `capital_gains_tax_rate` define the taxable position.

**Plain English:**
These four numbers describe what you own, today's price, your tax basis, and the tax rate.

**This answers the question:** "What is the position being tested?"

Example:
35 shares at 350 with basis 124 and 26% tax produces a sell-today after-tax value of 10,193.40.

`stop_loss_drops` contains candidate stop-loss triggers as percentage drops from today's price.

**Plain English:**
These are the loss limits you want to test.

**This answers the question:** "At what drops would I sell?"

Example:
`0.20` means sell if price drops 20%, which is 280 when today's price is 350.

`benchmark_recovery_tolerance` is the maximum rebound from a stop price that you consider tolerable for the highlighted benchmark stop.

**Plain English:**
This is your personal limit for how big a comeback you are willing to require.

**This answers the question:** "Which stop is still reasonable under my recovery tolerance?"

Example:
If tolerance is 30%, a 20% stop is acceptable because it needs a 25% rebound, but a 25% stop is not because it needs about 33%.

In [ ]:
from inputs import (
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
    stop_loss_drops,
    benchmark_recovery_tolerance,
)

In [ ]:
pd.Series(
    {
        "Shares held": shares,
        "Current price": f"${current_price:,.2f}",
        "Cost basis per share": f"${cost_basis_per_share:,.2f}",
        "Capital gains tax rate": f"{capital_gains_tax_rate:.0%}",
        "Stop-loss levels tested": ", ".join(f"{d:.0%}" for d in stop_loss_drops),
        "Benchmark recovery tolerance": f"{benchmark_recovery_tolerance:.0%}",
    },
    name="Inputs",
)

## Baseline and Stop Benchmark

Builds the sell-today benchmark and the stop-loss benchmark table.

**Plain English:**
It calculates the sell-now number once, then checks each stop level against it.

**This answers the question:** "What happens if each stop executes exactly at its stop price?"

Example:
At a 20% stop, price is 280 and the after-tax stopped value is 8,380.40.

In [ ]:
baseline = sell_today_baseline(
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)
stop_benchmark_df = build_stop_benchmark(
    stop_loss_drops,
    shares,
    current_price,
    cost_basis_per_share,
    capital_gains_tax_rate,
)

baseline

### Stop Benchmark Column Explanations

`stop_loss_drop` is the candidate stop trigger as a drop from today's price.

**Plain English:**
This is the percentage fall that causes a sale.

**This answers the question:** "How far down is the stop?"

Example:
`0.20` means 20% down.

`stop_price` is the price where that stop would execute in this simplified model.

**Plain English:**
This is the sale price assumed for the stop.

**This answers the question:** "At what price would the stop sell?"

Example:
20% below 350 is 280.

`after_tax_value_if_stopped` is the after-tax cash value if sold exactly at the stop price.

**Plain English:**
This is the cash you keep if the stop hits.

**This answers the question:** "How much money do I have after the stop sale and tax?"

Example:
At a 20% stop, this is 8,380.40.

`after_tax_loss_vs_selling_today` compares stopping at that price against selling today.

**Plain English:**
This is how much worse the stop sale is than selling immediately.

**This answers the question:** "How much do I lose by waiting until this stop?"

Example:
At a 20% stop, this is -1,813.00.

`required_recovery_from_stop_to_match_selling_today` is the rebound needed from the stop price to equal selling today after tax.

**Plain English:**
This says how big the comeback must be from the stop price.

**This answers the question:** "How much must the price recover after this stop level to catch up?"

Example:
From 280 to 350 requires a 25% rebound.

In [ ]:
stop_benchmark_df

## Best Benchmark Stop

Highlights the deepest tested stop where the required rebound from the stop price is no more than `benchmark_recovery_tolerance`.

**Plain English:**
It picks the deepest stop that still does not require too big of a comeback.

**This answers the question:** "Which tested stop is the deepest one I can tolerate under my rebound limit?"

Example:
With a 30% rebound tolerance, the 20% stop is selected because it needs a 25% rebound; the 25% stop needs about 33%.

In [ ]:
eligible_stops = stop_benchmark_df[
    stop_benchmark_df["required_recovery_from_stop_to_match_selling_today"] <= benchmark_recovery_tolerance
]

if eligible_stops.empty:
    best_benchmark_stop = stop_benchmark_df.head(1).iloc[0]
    benchmark_note = (
        f"No tested stop has required recovery at or below {benchmark_recovery_tolerance:.0%}. "
        "Showing the tightest tested stop instead."
    )
else:
    best_benchmark_stop = eligible_stops.tail(1).iloc[0]
    benchmark_note = f"Deepest tested stop where required recovery is at or below {benchmark_recovery_tolerance:.0%}."

pd.Series({
    "benchmark_rule": benchmark_note,
    "best_benchmark_stop_loss_drop": best_benchmark_stop["stop_loss_drop"],
    "best_benchmark_stop_price": best_benchmark_stop["stop_price"],
    "after_tax_value_if_stopped": best_benchmark_stop["after_tax_value_if_stopped"],
    "after_tax_loss_vs_selling_today": best_benchmark_stop["after_tax_loss_vs_selling_today"],
    "required_recovery_from_stop_to_match_selling_today": best_benchmark_stop["required_recovery_from_stop_to_match_selling_today"],
})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = stop_benchmark_df["stop_loss_drop"] * 100
y = stop_benchmark_df["required_recovery_from_stop_to_match_selling_today"] * 100
ax.bar(x, y, width=3, color="#4C78A8")
ax.axhline(benchmark_recovery_tolerance * 100, color="red", linestyle="--", label=f"Tolerance: {benchmark_recovery_tolerance:.0%}")
ax.set_title("Recovery Needed After Each Stop-Loss Level to Match Selling Today")
ax.set_xlabel("Stop-loss trigger: drop from today's price (%)")
ax.set_ylabel("Required rebound from the stop price (%)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.show()